Last run: Aug 8

In [ ]:
import os

os.environ.get("CONDA_DEFAULT_ENV")

### fitting Ks mag, Teff, Fe/H, and parallax

In [ ]:
from jaxstar import mistfit
from jax import random
import arviz, corner
import pandas as pd

In [ ]:
%matplotlib inline

In [ ]:
mf = mistfit.MistFit()

In [ ]:
from urllib.request import urlopen
import pandas as pd
import json

def get_tfop_info(target_name: str) -> dict:
    base_url = "https://exofop.ipac.caltech.edu/tess"
    url = f"{base_url}/target.php?id={target_name.replace(' ','')}&json"
    response = urlopen(url)
    assert response.code == 200, "Failed to get data from ExoFOP-TESS"
    try:
        data_json = json.loads(response.read())
        return data_json
    except Exception as e:
        print(e)
        raise ValueError(f"No TIC data found for {target_name}")


def get_params_from_tfop(tfop_info, name="planet_parameters", idx=None):
    params_dict = tfop_info.get(name)
    if idx is None:
        key = "pdate" if name == "planet_parameters" else "sdate"
        # get the latest parameter based on upload date
        dates = []
        for d in params_dict:
            t = d.get(key)
            dates.append(t)
        df = pd.DataFrame({"date": dates})
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        idx = df["date"].idxmax()
    return params_dict[idx]

In [ ]:
target = "TOI-5671"
tfop_info = get_tfop_info(target)
star_params = get_params_from_tfop(tfop_info, name="stellar_parameters")
planet_params = get_params_from_tfop(tfop_info, name="planet_parameters")

In [ ]:
mags = tfop_info["magnitudes"]
for i in range(len(mags)):
    print(i,mags[i]['band'])

In [ ]:
mags[1]0

In [ ]:
# observed Kmag, Teff, Fe/H, parallax (mas)
kmag_obs, kmag_err = 12.362, 0.023 #TIC v8
teff_obs, teff_err = 3278, 124 #IRD
feh_obs, feh_err = 0.31, 0.33 #IRD
parallax_obs, parallax_err = 6.07, 0.03 #mas

In [ ]:
mf.set_data(['kmag', 'teff', 'feh', 'parallax'],\
            [kmag_obs, teff_obs, feh_obs, parallax_obs],\
            [kmag_err, teff_err, feh_err, parallax_err])

In [ ]:
# setup HMC
ns = 10_000
mf.setup_hmc(num_warmup=ns, num_samples=ns)

In [ ]:
# run
rng_key = random.PRNGKey(0)
mf.run_hmc(rng_key, linear_age=True, flat_age_marginal=False, nodata=False)

In [ ]:
mcmc = mf.mcmc
samples = mf.samples

In [ ]:
# trace plot
fig = arviz.plot_trace(mcmc, var_names=mf.obskeys)

In [ ]:
# keys and observed values for corner plot
outkeys = mf.obskeys + ['mass', 'radius', 'age', 'eep']
obsvals = mf.obsvals + [None] * 4

In [ ]:
params = arviz.from_dict(dict(zip(outkeys, [samples[k] for k in outkeys])))
fig = corner.corner(params, show_titles="%.2f", truths=obsvals)